In [62]:
import torch
import gpytorch
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
random_seed = 42

In [63]:
def create_barron_dataset(
        D: int = 1,
        N_train: int = 500,
        N_test: int = 2000,
    ):
    """Generate multi-dimensional Barron function dataset for testing approximation algorithms.
    
    Args:
        D: Input dimension (default: 1)
        N_train: Number of training samples (default: 500)
        N_test: Number of test samples (default: 2000)
    
    Returns:
        Tuple of (X_train, X_test, y_train, y_test) tensors
    """
    torch.manual_seed(random_seed)
    
    # Fixed constant vector a, shape (D,)
    a = torch.tensor([2*j/D - 1 for j in range(1, D+1)], dtype=torch.float32)
    
    # Input points uniformly sampled from [-1, 1]^D
    X_train = torch.rand(N_train, D) * 2 - 1
    X_test  = torch.rand(N_test,  D) * 2 - 1
    
    # Target function: f(x) = sqrt(3/2) * (||x - a|| - ||x + a||)
    # The Barron norm of f is equal to one for all input dimensions
    def barron(X):
        return torch.sqrt(torch.tensor(1.5)) * (
            torch.norm(X - a, dim=1) - torch.norm(X + a, dim=1)
        )
    
    y_train = barron(X_train)
    y_test  = barron(X_test)
    
    print(f"[DEBUG] Barron dataset created: D={D}, N_train={N_train}, N_test={N_test}, y_range=[{y_train.min():.3f}, {y_train.max():.3f}]")
    
    return X_train, X_test, y_train, y_test

In [64]:
# ─── Exact GP (for small datasets, exact inference) ───────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    """Exact Gaussian Process for small-to-medium datasets.
    
    Uses exact inference with RBF kernel and ARD (automatic relevance determination).
    """
    def __init__(self, X_train, y_train, likelihood):
        """Initialize ExactGPModel.
        
        Args:
            X_train: Training input tensor, shape (N, D)
            y_train: Training output tensor, shape (N,)
            likelihood: GPyTorch likelihood object
        """
        super().__init__(X_train, y_train, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        # RBF kernel with ARD: separate lengthscale per input dimension
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=X_train.shape[1])
        )

    def forward(self, x):
        """Forward pass: compute mean and covariance."""
        mean = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar) # type: ignore

In [65]:
# ─── Sparse GP (for large datasets, inducing point approximation) ─────────────
class SparseGPModel(gpytorch.models.ApproximateGP):
    """Sparse Gaussian Process using variational inference.
    
    Uses inducing point approximation for scalability to large datasets.
    Inducing points are initialized randomly and learned during training.
    """
    def __init__(self, X_train, num_inducing=100):
        """Initialize SparseGPModel.
        
        Args:
            X_train: Training input tensor, shape (N, D)
            num_inducing: Number of inducing points (default: 100)
        """
        # Select random subset of training data as inducing points
        inducing_idx = torch.randperm(X_train.shape[0])[:num_inducing]
        inducing_points = X_train[inducing_idx]

        # Variational distribution q(u) over inducing points
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            num_inducing_points=num_inducing
        )
        # Variational strategy: defines how GP approximation is constructed
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True  # inducing points are optimized during training
        )
        super().__init__(variational_strategy)

        self.mean_module = gpytorch.means.ConstantMean()
        # RBF kernel with ARD: separate lengthscale per input dimension
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=X_train.shape[1])
        )

    def forward(self, x):
        """Forward pass: compute mean and covariance."""
        mean = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar) # type: ignore

In [66]:
def init_gp(gp_type, X_train, y_train, num_inducing=100):
    """Initialize a Gaussian Process model.
    
    Args:
        gp_type: Type of GP - 'exact' or 'sparse'
        X_train: Training input tensor, shape (N, D)
        y_train: Training output tensor, shape (N,)
        num_inducing: Number of inducing points for sparse GP (default: 100)
    
    Returns:
        model: GPyTorch model (ExactGPModel or SparseGPModel)
        likelihood: GPyTorch likelihood object
    
    Raises:
        ValueError: If gp_type is not 'exact' or 'sparse'
    """
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    
    if gp_type == 'exact':
        model = ExactGPModel(X_train, y_train, likelihood)
        print(f"[DEBUG] Initialized ExactGPModel: X_train.shape={X_train.shape}")
    elif gp_type == 'sparse':
        model = SparseGPModel(X_train, num_inducing=num_inducing)
        print(f"[DEBUG] Initialized SparseGPModel: X_train.shape={X_train.shape}, num_inducing={num_inducing}")
    else:
        raise ValueError(f"gp_type must be 'exact' or 'sparse', got '{gp_type}'")

    return model, likelihood

In [67]:
def train_gp(model, likelihood, X_train, y_train, num_iters=100, lr=0.1):
    """Train a Gaussian Process model.
    
    Args:
        model: GPyTorch model (ExactGPModel or SparseGPModel)
        likelihood: GPyTorch likelihood object
        X_train: Training input tensor, shape (N, D)
        y_train: Training output tensor, shape (N,)
        num_iters: Number of training iterations (default: 100)
        lr: Learning rate for Adam optimizer (default: 0.1)
    
    Returns:
        model: Trained model in eval mode
        likelihood: Trained likelihood in eval mode
    """

    # Combine model and likelihood parameters for optimization
    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(likelihood.parameters()), lr=lr
    )

    # Choose loss function based on GP type
    if isinstance(model, ExactGPModel):
        mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
        loss_name = "MLL"
    else:
        mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=X_train.shape[0])
        loss_name = "ELBO"

    print(f"[DEBUG] Training {type(model).__name__}: {num_iters} iters, lr={lr}, loss={loss_name}")

    # Training loop
    for i in range(num_iters):
        optimizer.zero_grad()
        output = model(X_train)
        loss = -mll(output, y_train) # type: ignore
        loss.backward()
        optimizer.step()
        
        if (i + 1) % max(1, num_iters // 5) == 0:
            print(f"  Iter {i+1}/{num_iters}, Loss: {loss.item():.6f}")
    
    print(f"[DEBUG] Training complete.")
    print(f"  Lengthscale: {model.covar_module.base_kernel.lengthscale.detach()}")
    print(f"  Outputscale: {model.covar_module.outputscale.item():.4f}")
    print(f"  Noise: {likelihood.noise.item():.6f}")
    print(f"  Mean const: {model.mean_module.constant.item():.4f}") # type: ignore

    return model, likelihood

In [68]:
# ─────────────────────────────────────────────────────────────
# Prediction and Evaluation Functions
# ─────────────────────────────────────────────────────────────

def predict(model, likelihood, X):
    """Make predictions using trained GP model.
    
    Args:
        model: Trained GPyTorch model
        likelihood: Trained likelihood object
        X: Input tensor, shape (N, D)
    
    Returns:
        mean: Predicted mean, shape (N,)
        std: Predicted standard deviation, shape (N,)
    """
    with torch.no_grad():
        # Query model and likelihood
        output = likelihood(model(X))
        mean = output.mean
        std = output.stddev
    
    return mean, std


def compute_mse(y_pred, y_true):
    """Compute Mean Squared Error.
    
    Args:
        y_pred: Predicted values, shape (N,)
        y_true: Ground truth values, shape (N,)
    
    Returns:
        mse: MSE value (scalar)
    """
    mse = ((y_pred - y_true) ** 2).mean()
    return mse


def compute_relative_l2(y_pred, y_true):
    """Compute relative L2 error: ||y_pred - y_true||_2 / ||y_true||_2
    
    Args:
        y_pred: Predicted values, shape (N,)
        y_true: Ground truth values, shape (N,)
    
    Returns:
        rel_l2: Relative L2 error (scalar)
    """
    rel_l2 = torch.norm(y_pred - y_true) / torch.norm(y_true)
    return rel_l2

In [69]:
def freeze_gp(model, likelihood):
    """Freeze (switch to eval mode) GP model and likelihood.
    
    Sets model and likelihood to evaluation mode, disabling gradient computation
    and dropout. Use this before making predictions or after training is complete.
    
    Args:
        model: GPyTorch model
        likelihood: GPyTorch likelihood object
    """
    model.eval()
    likelihood.eval()
    print(f"[DEBUG] {type(model).__name__} and likelihood frozen (eval mode)")


def unfreeze_gp(model, likelihood):
    """Unfreeze (switch to train mode) GP model and likelihood.
    
    Sets model and likelihood to training mode, enabling gradient computation.
    Use this before training.
    
    Args:
        model: GPyTorch model
        likelihood: GPyTorch likelihood object
    """
    model.train()
    likelihood.train()
    print(f"[DEBUG] {type(model).__name__} and likelihood unfrozen (train mode)")

In [70]:
def evaluate_gp(model, likelihood, X_test, y_test, X_train=None, y_train=None, model_name="GP"):
    """Evaluate GP model on test data and optionally on training data.
    
    Computes MSE and relative L2 error. If training data is provided, also evaluates
    on training set to detect overfitting (large gap indicates overfitting).
    
    Args:
        model: Trained GPyTorch model
        likelihood: Trained likelihood object
        X_test: Test input tensor, shape (N, D)
        y_test: Test output tensor, shape (N,)
        X_train: Optional training input tensor for overfitting check (default: None)
        y_train: Optional training output tensor for overfitting check (default: None)
        model_name: Name for printing (default: "GP")
    
    Returns:
        metrics: Dictionary with 'test_mse', 'test_rel_l2', and optionally 'train_mse', 'train_rel_l2'
    """
    # Switch mode
    freeze_gp(model, likelihood)
    
    # Evaluate on test set
    y_pred_test, _ = predict(model, likelihood, X_test)
    mse_test = compute_mse(y_pred_test, y_test)
    rel_l2_test = compute_relative_l2(y_pred_test, y_test)
    
    metrics = {
        'test_mse': mse_test.item(),
        'test_rel_l2': rel_l2_test.item()
    }
    
    # Print test results
    print(f"\n{model_name} Evaluation:")
    print(f"  Test MSE: {mse_test.item():.6f}")
    print(f"  Test Relative L2: {rel_l2_test.item():.6f}")
    
    # Evaluate on training set if provided (to detect overfitting)
    if X_train is not None and y_train is not None:
        y_pred_train, _ = predict(model, likelihood, X_train)
        mse_train = compute_mse(y_pred_train, y_train)
        rel_l2_train = compute_relative_l2(y_pred_train, y_train)
        
        metrics['train_mse'] = mse_train.item()
        metrics['train_rel_l2'] = rel_l2_train.item()
        
        # Print training results and overfitting gap
        print(f"  Train MSE: {mse_train.item():.6f} (gap: {(mse_test.item() - mse_train.item()):.6f})")
        print(f"  Train Relative L2: {rel_l2_train.item():.6f} (gap: {(rel_l2_test.item() - rel_l2_train.item()):.6f})")
        
        # Warn if significant overfitting detected
        if mse_test.item() > 2 * mse_train.item():
            print(f"  ⚠ Warning: Large overfitting gap detected (test MSE >> train MSE)")
    
    return metrics

In [71]:
X_train.shape, y_train.shape

(torch.Size([500, 1]), torch.Size([500]))

In [72]:
def sample_candidate_pairs(X_train, y_train, M=None, random_seed=42):
    """
    Sample M candidate pairs from training data using the SWIM delta trick.
    Guarantees idx_from != idx_to for all pairs.
    
    Args:
        X_train:     (N, D) tensor
        y_train:     (N,)   tensor
        M:           number of candidate pairs (defaults to N if None)
        random_seed: for reproducibility
    
    Returns:
        x_a: (M, D) — start points of pairs
        x_b: (M, D) — end points of pairs
        y_a: (M,)   — target values at x_a
        y_b: (M,)   — target values at x_b
    """
    N = X_train.shape[0]
    M = M if M is not None else N # update with ceiling later

    rng      = np.random.default_rng(random_seed)
    idx_from = rng.integers(low=0,   high=N,   size=M)
    delta    = rng.integers(low=1,   high=N-1, size=M)  # delta >= 1 guarantees no self-pairs
    idx_to   = (idx_from + delta) % N

    x_a = X_train[idx_from]  # (M, D)
    x_b = X_train[idx_to]    # (M, D)
    y_a = y_train[idx_from]  # (M,)
    y_b = y_train[idx_to]    # (M,)

    return x_a, x_b, y_a, y_b

In [73]:
def create_interior_points(x_a, x_b, T=30):
    """
    Create T interior points along each pair segment (x_a, x_b).
    Points are evenly spaced, excluding endpoints.
    
    Args:
        x_a: (M, D) — start points
        x_b: (M, D) — end points
        T:   number of interior points per pair
    
    Returns:
        x_interior:      (M, T, D) — interior points per pair
        x_interior_flat: (M*T, D)  — flattened for GP query
    """
    M = x_a.shape[0]

    # t in {1/(T+1), 2/(T+1), ..., T/(T+1)} — avoids endpoints
    t_values = torch.linspace(0, 1, T + 2)[1:-1]  # (T,)

    # x_a[:, None, :] broadcasts (M,1,D) + (1,T,1)*(M,1,D) → (M,T,D)
    x_interior = (
        x_a.unsqueeze(1) +
        t_values.view(1, T, 1) * (x_b - x_a).unsqueeze(1)
    )  # (M, T, D)

    x_interior_flat = x_interior.reshape(M * T, -1)  # (M*T, D)

    return x_interior, x_interior_flat

In [74]:
def compute_gp_swim_scores(model, likelihood, x_a, x_b, T=30, epsilon=1e-6):
    """
    Compute GP-SWIM pair scores using GP posterior gradients and uncertainty.

    The score for each pair (x_a, x_b) is:
        score = ||grad_mu(x_a) - grad_mu(x_b)||_inf  /  (std(x_a) + sum(std(interior)) + std(x_b))

    High score = large gradient difference (function varies a lot along this pair)
                 relative to low uncertainty (GP is confident about this region)

    Args:
        model:      frozen GP model (in eval mode)
        likelihood: frozen GP likelihood (in eval mode)
        x_a:        (M, D) — start points of candidate pairs
        x_b:        (M, D) — end points of candidate pairs
        T:          number of interior points per pair (default: 3)
        epsilon:    numerical stability constant (default: 1e-6)

    Returns:
        scores: (M,) — raw unnormalized scores per pair
        probs:  (M,) — normalized probabilities (sums to 1)
    """
    M = x_a.shape[0]

    # ── Step 1: Interior points for uncertainty along segment ──────────────────
    x_interior, x_interior_flat = create_interior_points(x_a, x_b, T=T)

    # Query GP at interior points (no grad needed — only for std)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_interior = likelihood(model(x_interior_flat))
        std_interior  = pred_interior.variance.sqrt().reshape(M, T)  # (M, T)

    # ── Step 2: Endpoint gradients through GP posterior mean ──────────────────
    # requires_grad=True so autograd can flow through GP mean
    x_a_g = x_a.detach().requires_grad_(True)  # (M, D)
    x_b_g = x_b.detach().requires_grad_(True)  # (M, D)

    with gpytorch.settings.fast_pred_var():
        pred_a = likelihood(model(x_a_g))
        pred_b = likelihood(model(x_b_g))

        mu_a  = pred_a.mean           # (M,)
        std_a = pred_a.variance.sqrt()  # (M,)
        mu_b  = pred_b.mean           # (M,)
        std_b = pred_b.variance.sqrt()  # (M,)

    # Gradient of GP posterior mean w.r.t. input points
    grad_a = torch.autograd.grad(mu_a.sum(), x_a_g)[0]  # (M, D)
    grad_b = torch.autograd.grad(mu_b.sum(), x_b_g)[0]  # (M, D)

    # ── Step 3: Score = gradient difference / uncertainty ─────────────────────
    numerator   = (grad_a - grad_b).abs().max(dim=1).values          # (M,) L-inf norm
    denominator = std_a + std_interior.sum(dim=1) + std_b + epsilon  # (M,)

    scores = numerator / denominator   # (M,)
    probs  = scores / scores.sum()     # (M,) sums to 1

    print(f"[DEBUG] Scores: min={scores.min():.4f}, max={scores.max():.4f}, mean={scores.mean():.4f}")
    print(f"[DEBUG] Probs:  min={probs.min():.6f},  max={probs.max():.6f},  sum={probs.sum():.6f}")

    return scores, probs

In [75]:
def select_pairs(x_a, x_b, probs, layer_width, random_seed=42):
    """
    Select layer_width winning pairs from candidates using GP-SWIM probabilities.
    
    Args:
        x_a:         (M, D) — candidate start points
        x_b:         (M, D) — candidate end points
        probs:       (M,)   — sampling probabilities (must sum to 1)
        layer_width: number of pairs to select (= number of neurons/edges)
        random_seed: for reproducibility
    
    Returns:
        x_a_selected: (layer_width, D) — selected start points
        x_b_selected: (layer_width, D) — selected end points
        selected_idx: (layer_width,)   — indices into original candidate arrays
    """
    M = x_a.shape[0]

    rng      = np.random.default_rng(random_seed)
    probs_np = probs.detach().cpu().numpy()

    selected_idx = rng.choice(
        M,                   # sample from M candidates
        size=layer_width,    # pick layer_width winners
        replace=True,        # same pair can be selected multiple times
        p=probs_np
    )

    x_a_selected = x_a[selected_idx]  # (layer_width, D)
    x_b_selected = x_b[selected_idx]  # (layer_width, D)

    print(f"[DEBUG] Selected {layer_width} pairs from {M} candidates")
    print(f"[DEBUG] Unique pairs selected: {len(set(selected_idx))} / {layer_width}")

    return x_a_selected, x_b_selected, selected_idx

In [76]:
def sample_edge_functions(model, likelihood, x_a_selected, x_b_selected, T_sample=200):
    """
    Sample GP posterior mean functions over each selected pair segment.
    These become the edge activation functions in the KAN-style network.

    Uses GP posterior MEAN (not rsample) to avoid noise and ensure
    smooth, reproducible edge functions across all edges.

    Args:
        model:          frozen GP model (eval mode)
        likelihood:     frozen GP likelihood (eval mode)
        x_a_selected:   (layer_width, D) — selected start points
        x_b_selected:   (layer_width, D) — selected end points
        T_sample:       number of points per segment (default: 200)

    Returns:
        x_segments:       (layer_width, T_sample, D) — segment input points
        edge_functions:   (layer_width, T_sample)    — GP mean values along each segment
    """
    layer_width = x_a_selected.shape[0]

    # ── Create dense points along each selected segment ────────────────────────
    # Reuse create_interior_points but with T_sample points including endpoints
    t_dense = torch.linspace(0, 1, T_sample)  # (T_sample,) — includes 0 and 1

    x_segments = (
        x_a_selected.unsqueeze(1) +
        t_dense.view(1, T_sample, 1) * (x_b_selected - x_a_selected).unsqueeze(1)
    )  # (layer_width, T_sample, D)

    # ── Query GP posterior mean per segment ────────────────────────────────────
    edge_functions = []

    for i in range(layer_width):
        x_seg_i = x_segments[i]  # (T_sample, D)

        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred_i = likelihood(model(x_seg_i))

        # Use posterior mean instead of rsample() to avoid:
        # 1. Stochastic noise in the lookup table
        # 2. Independent sampling inconsistency across edges
        # 3. Compounding errors in interpolation step
        f_mean = pred_i.mean  # (T_sample,)
        edge_functions.append(f_mean)

    edge_functions = torch.stack(edge_functions)  # (layer_width, T_sample)

    print(f"[DEBUG] x_segments shape:     {x_segments.shape}")
    print(f"[DEBUG] edge_functions shape: {edge_functions.shape}")
    print(f"[DEBUG] Value range: [{edge_functions.min():.4f}, {edge_functions.max():.4f}]")

    return x_segments, edge_functions

In [77]:
def interpolate_edge_functions(x_segments, edge_functions, X):
    """
    Interpolate edge functions at input points X using np.interp.
    

    Args:
        x_segments:     (layer_width, T_sample, D) — segment input points
        edge_functions: (layer_width, T_sample)    — GP mean values along segments
        X:              (N, D)                     — points to evaluate

    Returns:
        H: (N, layer_width) — feature matrix
    """
    layer_width = x_segments.shape[0]
    N           = X.shape[0]
    H           = torch.zeros(N, layer_width)

    x_np = X[:, 0].detach().numpy()  # (N,) — only first dim (1D only)

    for i in range(layer_width):
        seg_x = x_segments[i, :, 0].detach().numpy()   # (T_sample,)
        seg_f = edge_functions[i].detach().numpy()      # (T_sample,)

        H[:, i] = torch.tensor(np.interp(x_np, seg_x, seg_f))

    print(f"[DEBUG] H shape: {H.shape}")
    print(f"[DEBUG] H value range: [{H.min():.4f}, {H.max():.4f}]")
    print(f"[DEBUG] H rank (approx): {torch.linalg.matrix_rank(H)}")

    return H


def solve_output_layer(H_train, y_train):
    """
    Solve for output layer weights using OLS (least squares).
    Adds a bias column to H_train before solving.

    Args:
        H_train: (N_train, layer_width) — training feature matrix
        y_train: (N_train,)             — training targets

    Returns:
        W_out: (layer_width+1, 1) — output weights including bias
    """
    N_train = H_train.shape[0]

    # Add bias column
    H_train_b = torch.cat([H_train, torch.ones(N_train, 1)], dim=1)  # (N_train, layer_width+1)

    result = torch.linalg.lstsq(H_train_b, y_train.unsqueeze(1))
    W_out  = result.solution  # (layer_width+1, 1)

    print(f"[DEBUG] W_out shape: {W_out.shape}")

    return W_out


def predict_and_evaluate(H, y, W_out, split_name="Test"):
    """
    Predict and evaluate using solved output weights.

    Args:
        H:          (N, layer_width)      — feature matrix
        y:          (N,)                  — ground truth targets
        W_out:      (layer_width+1, 1)    — output weights including bias
        split_name: label for printing (default: "Test")

    Returns:
        y_pred: (N,)  — predictions
        mse:    scalar
        rel_l2: scalar
    """
    N = H.shape[0]

    # Add bias column
    H_b    = torch.cat([H, torch.ones(N, 1)], dim=1)  # (N, layer_width+1)
    y_pred = (H_b @ W_out).squeeze()                   # (N,)

    mse    = compute_mse(y_pred, y)
    rel_l2 = compute_relative_l2(y_pred, y)

    print(f"[{split_name}] MSE: {mse.item():.6f} | Relative L2: {rel_l2.item():.6f}")

    return y_pred, mse, rel_l2

In [82]:
D = 1 # number of features
T = 3
N_l = 56 # number of neurons of a hidden layer
grid = 20

# Stage 1: Create Barron dataset
X_train, X_test, y_train, y_test = create_barron_dataset(D=D)
print("\n")
# Stage 2: Fit Gaussian Process
model, likelihood = init_gp("exact", X_train, y_train)
model, likelihood = train_gp(model, likelihood, X_train, y_train)
freeze_gp(model, likelihood)
print("\n")
# Stage 3: Create M pairs from the training set like SWIM algorithm
x_a, x_b, y_a, y_b = sample_candidate_pairs(X_train, y_train)
print("\n")
# Stage 4: Create T interior point between x_a and x_b for posterior GP computation
x_interior, x_interior_flat = create_interior_points(x_a, x_b, T)
print("\n")
# Stage 5: Compute GP-SWIM like information gain probabilities
scores, probabilities = compute_gp_swim_scores(model, likelihood, x_a, x_b, T)
print("\n")
# Stage 6: Select the most informative data pairs using GP-SWIM like scores number of neuron times
x_a_selected, x_b_selected, selected_idx = select_pairs(
    x_a, x_b, probabilities, layer_width=N_l
)
print("\n")
print(x_a_selected.shape)
print(x_b_selected.shape)
print("\n")
# Stage 7: Sample GP posterior mean functions over each selected pair segment
# Edge activation functions in the KAN-style network
x_segments, edge_functions = sample_edge_functions(
    model, likelihood,
    x_a_selected, x_b_selected,
    T_sample=grid
)
print("\n")
print(x_segments.shape) 
print(edge_functions.shape) 
print("\n")
# Stage 8: interpolate values
H_train = interpolate_edge_functions(x_segments, edge_functions, X_train)
print("\n")

# Stage 9: Solve output layer on train only
W_out = solve_output_layer(H_train, y_train)
print("\n")

# Stage 10: Evaluate on both — test never touched W_out fitting
H_test  = interpolate_edge_functions(x_segments, edge_functions, X_test)
print("\n")
y_pred_train, mse_train, rel_l2_train = predict_and_evaluate(H_train, y_train, W_out, "Train")
y_pred_test,  mse_test,  rel_l2_test  = predict_and_evaluate(H_test,  y_test,  W_out, "Test")
print("\n")

[DEBUG] Barron dataset created: D=1, N_train=500, N_test=2000, y_range=[-2.449, 2.438]


[DEBUG] Initialized ExactGPModel: X_train.shape=torch.Size([500, 1])
[DEBUG] Training ExactGPModel: 100 iters, lr=0.1, loss=MLL
  Iter 20/100, Loss: -1.004704
  Iter 40/100, Loss: -2.908799
  Iter 60/100, Loss: -3.504992
  Iter 80/100, Loss: -3.564852
  Iter 100/100, Loss: -3.583437
[DEBUG] Training complete.
  Lengthscale: tensor([[2.3188]])
  Outputscale: 6.8509
  Noise: 0.000106
  Mean const: 0.0535
[DEBUG] ExactGPModel and likelihood frozen (eval mode)






[DEBUG] Scores: min=0.0001, max=0.6334, mean=0.1355
[DEBUG] Probs:  min=0.000001,  max=0.009351,  sum=1.000000


[DEBUG] Selected 56 pairs from 500 candidates
[DEBUG] Unique pairs selected: 49 / 56


torch.Size([56, 1])
torch.Size([56, 1])


[DEBUG] x_segments shape:     torch.Size([56, 20, 1])
[DEBUG] edge_functions shape: torch.Size([56, 20])
[DEBUG] Value range: [-2.3613, 2.4358]


torch.Size([56, 20, 1])
torch.Size([56, 20])


[DEBUG] H

In [83]:
# GP baseline
result = evaluate_gp(model, likelihood, X_test, y_test, X_train, y_train, "Exact GP")

[DEBUG] ExactGPModel and likelihood frozen (eval mode)

Exact GP Evaluation:
  Test MSE: 0.000000
  Test Relative L2: 0.000266
  Train MSE: 0.000000 (gap: -0.000000)
  Train Relative L2: 0.000264 (gap: 0.000003)
